# 01 — Data preparation and splitting

Clean the data, create the binary label, create the stratified full split, and create the fixed 200k/20k subset.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
##for path in [当前目录, 当前目录的上一级目录]
ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)

RAW = ROOT / "data/raw/train.csv"
CLEANED = ROOT / "data/processed/cleaned.csv"
FULL = ROOT / "data/splits/full"
SUBSET = ROOT / "data/splits/200k"
SEED = 42
##check if the raw data exists
assert RAW.exists()

## Clean and label

In [ ]:
# Read the required columns and remove rows with missing values
data = pd.read_csv(RAW, usecols=["id", "target", "comment_text"]).dropna(
    subset=["id", "target", "comment_text"]
)
# Clean comment text
data["comment_text"] = data["comment_text"].astype(str).str.strip()
# Remove empty and duplicate comments
data = data[data["comment_text"].ne("")].drop_duplicates("comment_text").copy()
# Convert the toxicity score to a binary label
data["label"] = data["target"].ge(0.5).astype("int8")
assert data["id"].is_unique 
CLEANED.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(CLEANED, index=False)
print(f"Cleaned rows: {len(data):,}")

## Stratified full split

In [ ]:
data = pd.read_csv(CLEANED)
# Create stratified train, validation and test splits
FULL.mkdir(parents=True, exist_ok=True)
full_paths = {name: FULL / f"{name}.csv" for name in ["train", "validation", "test"]}
train, held_out = train_test_split(
    data, test_size=0.20, random_state=SEED, stratify=data["label"]
)
validation, test = train_test_split(
    held_out, test_size=0.50, random_state=SEED, stratify=held_out["label"]
)
for name, frame in {"train": train, "validation": validation, "test": test}.items():
    frame[["id", "comment_text", "target", "label"]].sort_values("id").to_csv(
        full_paths[name], index=False
    )
full = {
    "train": pd.read_csv(full_paths["train"]),
    "validation": pd.read_csv(full_paths["validation"]),
    "test": pd.read_csv(full_paths["test"]),
}
## Check that there are no overlapping IDs between the splits
assert not (set(full["train"]["id"]) & set(full["validation"]["id"]))
assert not (set(full["train"]["id"]) & set(full["test"]["id"]))
assert not (set(full["validation"]["id"]) & set(full["test"]["id"]))
print({name: len(frame) for name, frame in full.items()})

## Fixed 200k/20k subset and checks

In [ ]:
SUBSET.mkdir(parents=True, exist_ok=True)
subset_paths = {name: SUBSET / f"{name}.csv" for name in ["train", "validation"]}
# Select the fixed 200k/20k training and validation subset
for name, rows in [("train", 200000), ("validation", 20000)]:
    selected = train_test_split(
        full[name], train_size=rows, random_state=SEED, stratify=full[name]["label"]
    )[0]
    selected.sort_values("id").to_csv(subset_paths[name], index=False)
subset = {
    "train": pd.read_csv(subset_paths["train"]),
    "validation": pd.read_csv(subset_paths["validation"]),
}
assert set(subset["train"]["id"]).issubset(set(full["train"]["id"]))
assert set(subset["validation"]["id"]).issubset(set(full["validation"]["id"]))
print("PASS | Full split and fixed subset checks completed.")